# Training a GFlowNet solver for the IHES Picture Cube

This notebook **trains a Generative Flow Network (GFlowNet)** to solve the
[CayleyPy SuperCube: IHES Picture Cube](https://www.kaggle.com/competitions/cayleypy-ihes-cube)
on a **TPU v3-8** with JAX + Equinox.

It implements *Learning Shortest Paths with Generative Flow Networks*
(Morozov, Maksimov, Tiapkin, Samsonov, 2026 — [arXiv:2603.01786](https://arxiv.org/abs/2603.01786)),
adapted to the IHES cube (72 facelets, 18 generators, group order ≈ 2.13·10²⁴).

## How GFN solving differs from a value/beam heuristic

The usual pipeline learns a **distance-to-solved** `V(s)` and beam-searches down it.
GFN instead flips the graph — the **solved state is the source**, every state is
terminal with uniform reward, so the partition function `Z = |group|` is known
exactly. Two policies are learned jointly:

* a **forward** policy that *scrambles* away from solved, and
* a **backward** policy `P_B` that *solves*.

A theorem (Morozov et al.) shows that **minimizing total flow forces `P_B` onto
shortest paths only** — geodesy falls out as a flow property, with *no distance
regression anywhere*. Training uses a regularized **trajectory-balance** loss.
At inference (companion beam notebook) you beam-search on cumulative `log P_B`.

## What this notebook produces

A trained `P_B` checkpoint (`train.eqx`) you can feed to the companion
**inference/beam notebook** to generate a submission. The exact hyperparameters
below were found by a sweep (`λ = 5·10⁻⁸`, `eps_explore = 0`); see the final
"Notes & honest results" cell for what to expect.

In [ ]:
# --- Setup: use Kaggle's PRE-INSTALLED jax/libtpu; add only pure-Python libs ---
# Do NOT pip-reinstall jax on a Kaggle TPU VM - it breaks the libtpu binding.
# Enable "TPU VM v3-8" + Internet in the notebook settings (right pane).
import os
os.environ["XLA_PYTHON_CLIENT_MEM_FRACTION"] = "0.95"
os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "2")

import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "equinox", "optax"],
               check=False)

import jax
print("jax", jax.__version__, "devices:", jax.device_count(),
      jax.devices()[0].device_kind)

In [ ]:
# --- Configuration (edit here) ---
# Hyperparameters below are the sweep-selected optimum for IHES (lnZ = 56.02).
# lambda (REG_COEF) is the one puzzle-specific knob: a clean single-peaked optimum
# at 5e-8 (5e-7 too strong -> over-regularizes; 5e-9 too weak). EPS_EXPLORE MUST be
# 0 for the cube: at this graph size any exploration collapses the policy (an
# eps=0.1 control gave a total 0/8 solve failure). See the paper's tuning tips.

CFG = dict(
    batch_global = 2048,     # trajectories per step (sharded over the 8 TPU cores)
    nmax         = 30,        # length of on-policy training trajectories
    iters        = 30000,    # SHORT demo (~20 min on v3-8) that produces a working
                             # (under-trained) checkpoint. The reference run used
                             # 800k (~4 h on a v6e-8) for near-converged lengths.
    hidden       = 2048,     # ResMLP width
    blocks       = 6,        # residual blocks
    emb_dim      = 16,       # per-facelet embedding
    lr           = 3e-4,
    clip         = 100.0,
    reg_coef     = 5e-8,     # lambda for the exact flow regularizer (paper eq. 8)
    eps_explore  = 0.0,      # MUST be 0 for cubes (see note above)
    log_every    = 2000,
    eval_every   = 20000,
    ckpt_every   = 20000,
    seed         = 0,
    save_dir     = "/kaggle/working",
)

# IHES Picture Cube group order (Schreier-Sims); lnZ is fixed, not learned.
IHES_ORDER = 2125922464947725402112000  # 2.1259e24  -> lnZ = 56.0162
for k, v in CFG.items():
    print(f"{k:14s} = {v}")

In [ ]:
# --- GFN core: environment, model, sampling, loss ---
# Self-contained port of Morozov et al. 2026 for a permutation Cayley graph.
import json, math
from pathlib import Path
from functools import partial
import numpy as np
import jax, jax.numpy as jnp
import equinox as eqx

NEG_INF = -1e30

# ---- environment (reversed-graph GFN construction, paper section 3.2) ----
def load_env(puzzle_info_path):
    info = json.load(open(puzzle_info_path, encoding="utf-8"))
    names = list(info["generators"].keys())
    gens = np.array([info["generators"][n] for n in names], dtype=np.int32)   # (A,S)
    solved = np.array(info["central_state"], dtype=np.int32)
    assert (solved == np.arange(len(solved))).all(), "solved must be identity"
    inv = lambda n: n[1:] if n.startswith("-") else "-" + n
    inv_idx = np.array([names.index(inv(n)) for n in names], dtype=np.int32)
    pre = gens[inv_idx].copy()                 # preimage of solved under each fwd action
    return gens, inv_idx, solved, pre

# ---- model: ResMLP trunk + backward/forward/stop logits (shared backbone) ----
class GFNPolicy(eqx.Module):
    emb: jax.Array
    inp: eqx.nn.Linear
    fcs: list
    lns: list
    outp: eqx.nn.Linear
    n_actions: int = eqx.field(static=True)
    def __init__(self, state_size, num_classes, n_actions, hidden, emb_dim, n_blocks, key):
        ks = jax.random.split(key, n_blocks + 3)
        self.n_actions = n_actions
        self.emb = jax.random.normal(ks[0], (num_classes, emb_dim)) * 0.02
        self.inp = eqx.nn.Linear(state_size * emb_dim, hidden, key=ks[1])
        self.fcs = [eqx.nn.Linear(hidden, hidden, key=ks[2 + i]) for i in range(n_blocks)]
        self.lns = [eqx.nn.LayerNorm((hidden,)) for _ in range(n_blocks)]
        self.outp = eqx.nn.Linear(hidden, n_actions + n_actions + 1, key=ks[-1])
    def __call__(self, x):
        h0 = self.inp(self.emb[x].reshape(-1)); h = h0
        for fc, ln in zip(self.fcs, self.lns):
            h = jax.nn.relu(fc(ln(h)) + h)
        out = self.outp(h + h0)
        return out[:self.n_actions], out[self.n_actions:]   # backward (A,), forward (A+1,)

# ---- sampling + loss, closed over the (device) environment tensors ----
def make_fns(gens, inv_idx, pre, solved, nmax, eps, true_log_z):
    gens_j, inv_j = jnp.asarray(gens), jnp.asarray(inv_idx)
    pre_j, solved_j = jnp.asarray(pre), jnp.asarray(solved)
    A = gens.shape[0]

    def entry_mask(s):                      # (N,A) True where a fwd move re-enters solved
        return (s[:, None, :] == pre_j[None]).all(-1)
    def apply_a(s, a):                      # apply gen a to each row
        return jnp.take_along_axis(s, gens_j[a], axis=1)

    def sample(key, model, batch):          # on-policy fwd trajectories from solved
        s0 = jnp.tile(solved_j[None], (batch, 1))
        def step(carry, _):
            k, s = carry
            _, fwd = jax.vmap(model)(s); logits = fwd[:, :A]
            k, ke, ka = jax.random.split(k, 3)
            logits = jnp.where(jax.random.bernoulli(ke, eps, (batch,))[:, None], 0.0, logits)
            logits = jnp.where(entry_mask(s), NEG_INF, logits)
            a = jax.random.categorical(ka, logits)
            return (k, apply_a(s, a)), (apply_a(s, a), a)
        (_, _), (st, ac) = jax.lax.scan(step, (key, s0), None, length=nmax)
        return jnp.concatenate([s0[None], st], 0), ac

    def loss_fn(params, static, states, actions, lam, flow_shift):
        model = eqx.combine(params, static)
        T, b, S = states.shape; flat = states.reshape(T * b, S)
        bwd, fwd = jax.vmap(model)(flat)
        mask = entry_mask(flat)
        full = jnp.concatenate([mask, jnp.zeros((mask.shape[0], 1), bool)], 1)
        log_pf = jax.nn.log_softmax(jnp.where(full, NEG_INF, fwd), -1).reshape(T, b, -1)
        log_pb = jax.nn.log_softmax(bwd, -1).reshape(T, b, -1)
        log_flows = -log_pf[..., -1]                       # F(s) = 1/P_F(stop|s)
        pf_t = jnp.take_along_axis(log_pf[:-1], actions[..., None], -1)[..., 0]
        pb_t = jnp.take_along_axis(log_pb[1:], inv_j[actions][..., None], -1)[..., 0]
        z = jnp.zeros((1, b))
        pre_f = jnp.concatenate([z, jnp.cumsum(pf_t, 0)], 0)
        pre_b = jnp.concatenate([z, jnp.cumsum(pb_t, 0)], 0)
        residual = true_log_z + pre_f - pre_b - log_flows   # prefix trajectory balance
        tb = jnp.mean(residual ** 2)
        # exact paper flow reg, float32 fixed-shift factorization (lnZ 56 fits fp32):
        lse = jax.scipy.special.logsumexp(log_flows[1:], 0)
        reg = lam * jnp.mean(jnp.exp(jnp.minimum(lse - flow_shift, 55.0)))
        return tb + reg, jnp.array([tb, reg, jnp.mean(log_flows[1]), jnp.mean(log_flows[-1])])

    def greedy(model, starts, max_steps):    # argmax P_B rollout (for a cheap eval)
        def step(carry, _):
            s, done, ln = carry
            a = jnp.argmax(jax.vmap(model)(s)[0], -1)
            s2 = jnp.where(done[:, None], s, apply_a(s, a))
            done2 = done | (s2 == solved_j[None]).all(-1)
            return (s2, done2, ln + (~done)), None
        d0 = (starts == solved_j[None]).all(-1)
        (s, d, ln), _ = jax.lax.scan(step, (starts, d0, jnp.zeros(starts.shape[0], jnp.int32)),
                                     None, length=max_steps)
        return d, ln

    return sample, loss_fn, greedy, apply_a

print("core defined")

In [ ]:
# --- Training loop (pmap data-parallel over the 8 TPU cores) ---
import os, time, optax

# puzzle_info.json ships in the companion dataset. Kaggle mount paths vary
# (/kaggle/input/<slug>/ vs /kaggle/input/datasets/<owner>/<slug>/), so glob for it.
import glob
cands = (glob.glob("/kaggle/input/**/puzzle_info.json", recursive=True))
assert cands, "puzzle_info.json not found - attach the ihes-gfn-checkpoint dataset"
PUZZLE = cands[0]
print("puzzle:", PUZZLE)

gens, inv_idx, solved, pre = load_env(PUZZLE)
true_log_z = math.log(IHES_ORDER)
ndev = jax.device_count(); b_local = CFG["batch_global"] // ndev
sample, loss_fn, greedy, apply_a = make_fns(gens, inv_idx, pre, solved, CFG["nmax"],
                                            CFG["eps_explore"], true_log_z)
print(f"devices={ndev} b_local={b_local} lnZ={true_log_z:.4f} actions={gens.shape[0]}")

key = jax.random.key(CFG["seed"]); key, mk = jax.random.split(key)
model = GFNPolicy(len(solved), len(solved), gens.shape[0], CFG["hidden"],
                  CFG["emb_dim"], CFG["blocks"], mk)
params, static = eqx.partition(model, eqx.is_array)
opt = optax.chain(optax.clip_by_global_norm(CFG["clip"]),
                  optax.adamw(CFG["lr"], weight_decay=1e-5))
opt_state = opt.init(params)

def loss_wrap(p, s, a, lam, sh):
    return loss_fn(p, static, s, a, lam, sh)

@partial(jax.pmap, axis_name="dp", static_broadcasted_argnums=())
def train_step(params, opt_state, key, lam, shift):
    key, sk = jax.random.split(key)
    st, ac = sample(sk, eqx.combine(params, static), b_local)
    (loss, met), g = jax.value_and_grad(loss_wrap, has_aux=True)(params, st, ac, lam, shift)
    g = jax.lax.pmean(g, "dp"); met = jax.lax.pmean(met, "dp")
    upd, opt_state = opt.update(g, opt_state, params)
    return optax.apply_updates(params, upd), opt_state, key, met

# Replicate across the 8 cores by broadcasting a leading device axis; pmap then
# distributes it (version-agnostic - avoids the deprecated device_put_replicated).
def rep(tree):
    return jax.tree_util.tree_map(
        lambda x: jnp.broadcast_to(jnp.asarray(x), (ndev,) + jnp.asarray(x).shape), tree)
P, O = rep(params), rep(opt_state)
keys = jax.random.split(jax.random.fold_in(key, 1), ndev)
lam_r, sh_r = rep(jnp.float32(CFG["reg_coef"])), rep(jnp.float32(0.0))

def save(tag, P):
    ph = jax.tree_util.tree_map(lambda x: x[0], P)
    m = eqx.combine(ph, static)
    eqx.tree_serialise_leaves(f"{CFG['save_dir']}/{tag}.eqx",
                              {"m": ph, "z": jnp.float32(true_log_z)})
    json.dump({"task": "ihes", "hidden": CFG["hidden"], "blocks": CFG["blocks"],
               "emb_dim": CFG["emb_dim"], "lnZ": true_log_z},
              open(f"{CFG['save_dir']}/config.json", "w"))

t0 = time.time()
for it in range(CFG["iters"]):
    P, O, keys, met = train_step(P, O, keys, lam_r, sh_r)
    if (it + 1) % CFG["log_every"] == 0:
        m = np.asarray(met[0])
        print(f"iter {it+1:>7} tb={m[0]:.3g} reg={m[1]:.3g} logF_d1={m[2]:.3g} "
              f"logF_last={m[3]:.3g}  {(it+1)/(time.time()-t0):.1f} it/s", flush=True)
    if (it + 1) % CFG["ckpt_every"] == 0:
        save("train", P); print("  checkpoint saved")
save("train", P)
print(f"done in {(time.time()-t0)/3600:.2f} h -> /kaggle/working/train.eqx")

In [ ]:
# --- Quick sanity eval: greedy P_B on exact-depth states ---
# BFS a few shells from solved and check greedy-P_B optimality. (Real solving uses
# the wide beam in the companion inference notebook; greedy only probes shallow.)
def bfs_shells(gens, solved, max_depth, per):
    rng = np.random.default_rng(0); seen = {solved.astype(np.uint8).tobytes()}
    frontier = solved[None].astype(np.uint8); out = {}
    for d in range(1, max_depth + 1):
        ch = np.unique(frontier[:, gens].reshape(-1, frontier.shape[1]), axis=0)
        keep = np.fromiter((c.tobytes() not in seen for c in ch), bool, len(ch))
        frontier = ch[keep]
        for c in frontier: seen.add(c.tobytes())
        idx = rng.choice(len(frontier), min(per, len(frontier)), replace=False)
        out[d] = frontier[idx].astype(np.int32)
    return out

ph = jax.tree_util.tree_map(lambda x: x[0], P)
mdl = eqx.combine(ph, static)
shells = bfs_shells(gens, solved, 4, 64)
for d, st in shells.items():
    done, ln = greedy(mdl, jnp.asarray(st), 4 * d + 8)
    opt = float(((ln == d) & done).mean())
    print(f"depth {d}: greedy solve {float(done.mean()):.2f}  optimal {opt:.2f}")
print("(shallow greedy is just a sanity check; wide-beam solving is the real metric)")

# Notes & honest results

**Output:** `/kaggle/working/train.eqx` (+ `config.json`) — feed these to the
companion **inference/beam notebook** (`cayleypy-ihes-gfn-beam`) to generate a
submission, or publish them as a dataset for others.

## What to expect (measured)

* The `iters = 200000` demo (~2–3 h on v3-8) gives a **working but under-trained**
  policy. The reference model (`800k` iters, v6e-8) beam-solves the whole test set
  and averages ≈ **25–26 moves/puzzle**.
* This is a **faithful GFN**, not a state-of-the-art solver. On the IHES cube a
  well-tuned GFN roughly **ties a strong value-guided beam** and lands a few
  percent above the near-optimal community solutions — consistent with the paper's
  own finding that GFN ≈ ties CayleyPy Cube. Its appeal is *method* (no distance
  regression; one forward pass scores all children) rather than raw score.

## Tuning tips (from a sweep)

* `reg_coef` (λ) is the one puzzle-specific knob and has a **single-peaked
  optimum** — here `5e-8`. Larger over-regularizes (fails to solve), smaller
  under-shortens. The paper's rule of thumb: *pick the largest λ that still solves*.
* `eps_explore` **must be 0** for cube-family puzzles. At this graph size any
  forward exploration collapses `P_B` (an `eps=0.1` control failed to solve
  anything). Exploration is only needed on much larger graphs.
* `nmax` (training trajectory length) can be well below the diameter and still
  generalize — the paper uses 24 for 3×3×3; 30 here.

## References

* Morozov, Maksimov, Tiapkin, Samsonov, *Learning Shortest Paths with Generative
  Flow Networks*, 2026 — [arXiv:2603.01786](https://arxiv.org/abs/2603.01786),
  code [github.com/GreatDrake/gfn-pathfinding](https://github.com/GreatDrake/gfn-pathfinding).
* Malkin et al., *Trajectory Balance*, NeurIPS 2022.